## Qwen2.5-VL Grounding Task

This tutorial demonstrates the end-to-end workflow for grounding tasks using Qwen2.5-VL. You can also use other multimodal models such as InternVL2.5 or Qwen2-VL.

We use the [AI-ModelScope/coco](https://modelscope.cn/datasets/AI-ModelScope/coco) dataset to demonstrate the complete workflow.

If you need to use a custom dataset, it should follow this format:

In [ ]:
{"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "<image>Describe the image"}, {"role": "assistant", "content": "<ref-object><bbox> and <ref-object><bbox> are playing on the beach"}], "images": ["/xxx/x.jpg"], "objects": {"ref": ["a dog", "a woman"], "bbox": [[331.5, 761.4, 853.5, 1594.8], [676.5, 685.8, 1099.5, 1427.4]]}}
{"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "<image>Find <ref-object> in the image"}, {"role": "assistant", "content": "<bbox><bbox>"}], "images": ["/xxx/x.jpg"], "objects": {"ref": ["sheep"], "bbox": [[90.9, 160.8, 135, 212.8], [360.9, 480.8, 495, 532.8]]}}
{"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "<image>Help me open Google Chrome"}, {"role": "assistant", "content": "Action: click(start_box='<bbox>')"}], "images": ["/xxx/x.jpg"], "objects": {"ref": [], "bbox": [[615, 226]]}}

When preprocessing datasets, ms-swift applies model-specific grounding formats: `ref` in `objects` fills `<ref-object>`, and `bbox` is normalized to 0-1000 or kept as raw coordinates depending on the model, filling `<bbox>`.

Before training, install the framework:


In [ ]:
git clone https://github.com/Vaibhava2711/Agentic-LLM-for-Autonomous-Data-Science.git
cd Agentic-LLM-for-Autonomous-Data-Science/deepanalyze/ms-swift
pip install -e .


Then, use the following command for training. See documentation for parameter descriptions.

### Training

Single GPU training:


In [ ]:
# GPU Memory: 24GiB
CUDA_VISIBLE_DEVICES=0 \
MAX_PIXELS=1003520 \
swift sft \
    --model Qwen/Qwen2.5-VL-7B-Instruct \
    --dataset 'AI-ModelScope/coco#2000' \
    --split_dataset_ratio 0.01 \
    --train_type lora \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --freeze_vit true \
    --gradient_accumulation_steps 16 \
    --eval_steps 100 \
    --save_steps 100 \
    --save_total_limit 5 \
    --logging_steps 5 \
    --max_length 2048 \
    --output_dir output \
    --warmup_ratio 0.05 \
    --dataloader_num_workers 4 \
    --dataset_num_proc 4


Then export the trained model checkpoint:


In [ ]:
swift export \
    --adapters output/vx-xxx/checkpoint-xxx \
    --push_to_hub true \
    --hub_model_id '<model-id>' \
    --hub_token '<sdk-token>' \
    --use_hf false

The trained checkpoint can be exported and saved.

### Inference

After training, run inference on the validation set:


In [ ]:
CUDA_VISIBLE_DEVICES=0 \
swift infer \
    --adapters swift/test_grounding \
    --stream true \
    --load_data_args true \
    --max_new_tokens 512 \
    --dataset_num_proc 4

Inference can also be run via Python code:


In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import re
from typing import Literal
from swift.llm import (
    PtEngine, RequestConfig, BaseArguments, InferRequest, safe_snapshot_download, draw_bbox, load_image, load_dataset, InferEngine
)
from IPython.display import display

def infer_stream(engine: InferEngine, infer_request: InferRequest):
    request_config = RequestConfig(max_tokens=512, temperature=0, stream=True)
    gen_list = engine.infer([infer_request], request_config)
    query = infer_request.messages[0]['content']
    print(f'query: {query}\nresponse: ', end='')
    response = ''
    for resp in gen_list[0]:
        if resp is None:
            continue
        delta = resp.choices[0].delta.content
        response += delta
        print(delta, end='', flush=True)
    print()
    return response

def draw_bbox_qwen2_vl(image, response, norm_bbox: Literal['norm1000', 'none']):
    matches = re.findall(
        r'<\|object_ref_start\|>(.*?)<\|object_ref_end\|><\|box_start\|>\((\d+),(\d+)\),\((\d+),(\d+)\)<\|box_end\|>',
        response)
    ref = []
    bbox = []
    for match_ in matches:
        ref.append(match_[0])
        bbox.append(list(match_[1:]))
    draw_bbox(image, ref, bbox, norm_bbox=norm_bbox)

# Download weights and load model
output_dir = 'images_bbox'
model_id_or_path = 'swift/test_grounding'
output_dir = os.path.abspath(os.path.expanduser(output_dir))
adapter_path = safe_snapshot_download(model_id_or_path)
args = BaseArguments.from_pretrained(adapter_path)
engine = PtEngine(args.model, adapters=[adapter_path])

# Load validation set and run inference
_, val_dataset = load_dataset(args.dataset, split_dataset_ratio=args.split_dataset_ratio, num_proc=4, seed=args.seed)
print(f'output_dir: {output_dir}')
os.makedirs(output_dir, exist_ok=True)
for i, data in enumerate(val_dataset):
    image = data['images'][0]
    image = load_image(image['bytes'] or image['path'])
    display(image)
    response = infer_stream(engine, InferRequest(**data))
    draw_bbox_qwen2_vl(image, response, norm_bbox=args.norm_bbox)
    print('-' * 50)
    image.save(os.path.join(output_dir, f'{i}.png'))
    display(image)